# W-2 Tax Form CBA Experiment

**Domain**: Tax Forms (W-2)  
**Method**: Vision extraction — LLM reads W-2 images and maps values to canonical concepts  
**Models**: Claude Haiku 4.5, GPT-4o-mini  
**Scoring**: Value-first CBA matching

W-2 forms have a fixed IRS-mandated layout with labeled boxes. This serves as a **control condition** —
concept binding should be straightforward because each box has a clear label and position.
We test 44 concepts across 8 families. Expected: low misbinding, confirming CBA doesn't falsely inflate errors.

Key confusable pairs: wages_tips_compensation ↔ social_security_wages ↔ medicare_wages (similar dollar amounts),
federal_tax_withheld ↔ social_security_tax ↔ medicare_tax (adjacent tax boxes),
employer_address ↔ employee_address (stacked address blocks).

In [1]:
import subprocess, sys
for pkg in ["anthropic", "openai", "datasets", "Pillow"]:
    try:
        __import__(pkg if pkg != "Pillow" else "PIL")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--break-system-packages", "-q"])

import anthropic, openai, json, time, os, random, io, base64
from PIL import Image as PILImage
from datasets import load_dataset
from collections import defaultdict, Counter
from google.colab import userdata


ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print("Setup complete")

Setup complete


In [2]:
# Load synthetic W-2 dataset from HuggingFace
ds = load_dataset("singhsays/fake-w2-us-tax-form-dataset")
train_data = ds["train"]
print(f"W-2 dataset loaded: {len(train_data)} forms")

# Inspect structure
sample = train_data[0]
gt_raw = json.loads(sample["ground_truth"]) if isinstance(sample["ground_truth"], str) else sample["ground_truth"]
gt_parse = gt_raw.get("gt_parse", gt_raw)
print(f"\nGround truth keys ({len(gt_parse)}):")
for k, v in list(gt_parse.items())[:10]:
    print(f"  {k}: {v}")
print(f"  ... ({len(gt_parse) - 10} more)")

# Field mapping: HuggingFace key -> (canonical_concept, family)
W2_FIELD_MAP = {
    "box_a_employee_ssn": ("employee_ssn", "identity"),
    "box_b_employer_identification_number": ("employer_ein", "identity"),
    "box_c_employer_name": ("employer_name", "identity"),
    "box_e_employee_name": ("employee_name", "identity"),
    "box_d_control_number": ("control_number", "identity"),
    "box_c_employer_street_address": ("employer_address", "address"),
    "box_c_employer_city_state_zip": ("employer_city_state_zip", "address"),
    "box_e_employee_street_address": ("employee_address", "address"),
    "box_e_employee_city_state_zip": ("employee_city_state_zip", "address"),
    "box_1_wages": ("wages_tips_compensation", "compensation"),
    "box_3_social_security_wages": ("social_security_wages", "compensation"),
    "box_5_medicare_wages": ("medicare_wages", "compensation"),
    "box_7_social_security_tips": ("social_security_tips", "compensation"),
    "box_8_allocated_tips": ("allocated_tips", "compensation"),
    "box_2_federal_tax_withheld": ("federal_tax_withheld", "withholding"),
    "box_4_social_security_tax_withheld": ("social_security_tax", "withholding"),
    "box_6_medicare_wages_tax_withheld": ("medicare_tax", "withholding"),
    "box_10_dependent_care_benefits": ("dependent_care_benefits", "benefits"),
    "box_11_nonqualified_plans": ("nonqualified_plans", "benefits"),
    "box_12a_code": ("box_12a_code", "box12"),
    "box_12a_value": ("box_12a_value", "box12"),
    "box_12b_code": ("box_12b_code", "box12"),
    "box_12b_value": ("box_12b_value", "box12"),
    "box_12c_code": ("box_12c_code", "box12"),
    "box_12c_value": ("box_12c_value", "box12"),
    "box_12d_code": ("box_12d_code", "box12"),
    "box_12d_value": ("box_12d_value", "box12"),
    "box_13_statutory_employee": ("statutory_employee", "box13"),
    "box_13_retirement_plan": ("retirement_plan", "box13"),
    "box_13_third_party_sick_pay": ("third_party_sick_pay", "box13"),
    "box_15_1_state": ("state_1", "state_local"),
    "box_15_1_employer_state_id": ("state_1_employer_id", "state_local"),
    "box_16_1_state_wages": ("state_1_wages", "state_local"),
    "box_17_1_state_income_tax": ("state_1_income_tax", "state_local"),
    "box_18_1_local_wages": ("local_1_wages", "state_local"),
    "box_19_1_local_income_tax": ("local_1_income_tax", "state_local"),
    "box_20_1_locality_name": ("local_1_name", "state_local"),
    "box_15_2_state": ("state_2", "state_local"),
    "box_15_2_employer_state_id": ("state_2_employer_id", "state_local"),
    "box_16_2_state_wages": ("state_2_wages", "state_local"),
    "box_17_2_state_income_tax": ("state_2_income_tax", "state_local"),
    "box_18_2_local_wages": ("local_2_wages", "state_local"),
    "box_19_2_local_income_tax": ("local_2_income_tax", "state_local"),
    "box_20_2_locality_name": ("local_2_name", "state_local"),
}

def convert_gt(gt_parse):
    """Convert HuggingFace GT to canonical concept format."""
    canonical = {}
    for hf_key, (concept_id, family) in W2_FIELD_MAP.items():
        val = gt_parse.get(hf_key)
        if val is None or val == "":
            canonical[concept_id] = "N/A"
        elif isinstance(val, bool):
            canonical[concept_id] = "Yes" if val else "No"
        elif isinstance(val, float):
            canonical[concept_id] = f"{val:.2f}"
        elif isinstance(val, int):
            canonical[concept_id] = str(val)
        else:
            canonical[concept_id] = str(val)
    return canonical

print(f"\nField mapping: {len(W2_FIELD_MAP)} HF fields -> canonical concepts")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/886 [00:00<?, ?B/s]

data/train-00000-of-00001-26677581e561d5(…):   0%|          | 0.00/279M [00:00<?, ?B/s]

data/validation-00000-of-00001-dec92c211(…):   0%|          | 0.00/15.5M [00:00<?, ?B/s]

data/test-00000-of-00001-d2b8d24cfd674b2(…):   0%|          | 0.00/15.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

W-2 dataset loaded: 1800 forms

Ground truth keys (45):
  box_b_employer_identification_number: 47-5592725
  box_c_employer_name: Bennett, Allen and Yang Inc
  box_c_employer_street_address: 40301 Cameron Village Suite 661
  box_c_employer_city_state_zip: Aguirrebury NH 36219-7671
  box_a_employee_ssn: 412-88-2525
  box_e_employee_name: Michele Hebert
  box_e_employee_street_address: 9888 Zimmerman Roads Apt. 425
  box_e_employee_city_state_zip: Moorestad MO 77456-6485
  box_d_control_number: 4390616
  box_1_wages: 141194.15
  ... (35 more)

Field mapping: 44 HF fields -> canonical concepts


In [3]:
W2_ONTOLOGY = {
    "families": {
        "identity": ["employee_ssn", "employer_ein", "employer_name", "employee_name", "control_number"],
        "address": ["employer_address", "employer_city_state_zip", "employee_address", "employee_city_state_zip"],
        "compensation": ["wages_tips_compensation", "social_security_wages", "medicare_wages", "social_security_tips", "allocated_tips"],
        "withholding": ["federal_tax_withheld", "social_security_tax", "medicare_tax"],
        "benefits": ["dependent_care_benefits", "nonqualified_plans"],
        "box12": ["box_12a_code", "box_12a_value", "box_12b_code", "box_12b_value",
                   "box_12c_code", "box_12c_value", "box_12d_code", "box_12d_value"],
        "box13": ["statutory_employee", "retirement_plan", "third_party_sick_pay"],
        "state_local": ["state_1", "state_1_employer_id", "state_1_wages", "state_1_income_tax",
                        "local_1_wages", "local_1_income_tax", "local_1_name",
                        "state_2", "state_2_employer_id", "state_2_wages", "state_2_income_tax",
                        "local_2_wages", "local_2_income_tax", "local_2_name"],
    }
}

ALL_CONCEPTS = [c for fam in W2_ONTOLOGY["families"].values() for c in fam]
CONCEPT_TO_FAMILY = {c: f for f, cs in W2_ONTOLOGY["families"].items() for c in cs}

W2_SYSTEM_PROMPT = """You are a document AI extraction system. You extract structured data from W-2 tax form images.

Given an image of a W-2 form, extract values for the following canonical concept keys.
You MUST use exactly these keys — do NOT use the box numbers or labels from the form.

CANONICAL CONCEPT KEYS:
- employee_ssn: Employee Social Security Number
- employer_ein: Employer Identification Number
- employer_name: Employer/company name
- employee_name: Employee full name
- control_number: Control number
- employer_address: Employer street address
- employer_city_state_zip: Employer city, state, and ZIP
- employee_address: Employee street address
- employee_city_state_zip: Employee city, state, and ZIP
- wages_tips_compensation: Total wages, tips, and other compensation
- social_security_wages: Wages subject to Social Security tax
- medicare_wages: Wages and tips subject to Medicare tax
- social_security_tips: Tips subject to Social Security tax
- allocated_tips: Allocated tips
- federal_tax_withheld: Federal income tax withheld
- social_security_tax: Social Security tax withheld
- medicare_tax: Medicare tax withheld
- dependent_care_benefits: Dependent care benefits
- nonqualified_plans: Nonqualified deferred compensation plans
- box_12a_code: First deferred compensation/benefit code
- box_12a_value: First deferred compensation/benefit amount
- box_12b_code: Second deferred compensation/benefit code
- box_12b_value: Second deferred compensation/benefit amount
- box_12c_code: Third deferred compensation/benefit code
- box_12c_value: Third deferred compensation/benefit amount
- box_12d_code: Fourth deferred compensation/benefit code
- box_12d_value: Fourth deferred compensation/benefit amount
- statutory_employee: Is the employee a statutory employee (Yes/No)
- retirement_plan: Is the employee in a retirement plan (Yes/No)
- third_party_sick_pay: Was third-party sick pay provided (Yes/No)
- state_1: Primary state abbreviation
- state_1_employer_id: Employer state ID (primary state)
- state_1_wages: State wages for primary state
- state_1_income_tax: State income tax withheld for primary state
- local_1_wages: Local wages for primary locality
- local_1_income_tax: Local income tax withheld for primary locality
- local_1_name: Primary locality name
- state_2: Secondary state abbreviation
- state_2_employer_id: Employer state ID (secondary state)
- state_2_wages: State wages for secondary state
- state_2_income_tax: State income tax withheld for secondary state
- local_2_wages: Local wages for secondary locality
- local_2_income_tax: Local income tax withheld for secondary locality
- local_2_name: Secondary locality name

RULES:
1. Return ONLY a JSON object with exactly these 44 keys.
2. Monetary values: numeric string WITHOUT $ or commas (e.g., "4523.67").
3. SSN format: XXX-XX-XXXX. EIN format: XX-XXXXXXX.
4. Boolean fields (checkboxes): "Yes" or "No".
5. If a field is not present or empty on the form, use "N/A".
6. Return ONLY valid JSON, no other text."""

print(f"Schema: {len(ALL_CONCEPTS)} concepts, {len(W2_ONTOLOGY['families'])} families")
for fam, concepts in W2_ONTOLOGY["families"].items():
    print(f"  {fam} ({len(concepts)}): {concepts[:3]}{'...' if len(concepts) > 3 else ''}")

Schema: 44 concepts, 8 families
  identity (5): ['employee_ssn', 'employer_ein', 'employer_name']...
  address (4): ['employer_address', 'employer_city_state_zip', 'employee_address']...
  compensation (5): ['wages_tips_compensation', 'social_security_wages', 'medicare_wages']...
  withholding (3): ['federal_tax_withheld', 'social_security_tax', 'medicare_tax']
  benefits (2): ['dependent_care_benefits', 'nonqualified_plans']
  box12 (8): ['box_12a_code', 'box_12a_value', 'box_12b_code']...
  box13 (3): ['statutory_employee', 'retirement_plan', 'third_party_sick_pay']
  state_local (14): ['state_1', 'state_1_employer_id', 'state_1_wages']...


In [4]:
SAMPLE_SIZE = 50
random.seed(42)

IMG_DIR = "/tmp/w2_images"
os.makedirs(IMG_DIR, exist_ok=True)

indices = list(range(len(train_data)))
random.shuffle(indices)
selected = sorted(indices[:SAMPLE_SIZE])

samples = []
for i, idx in enumerate(selected):
    rec = train_data[idx]
    doc_id = f"w2_{i:04d}"

    # Parse and convert ground truth
    gt_raw = json.loads(rec["ground_truth"]) if isinstance(rec["ground_truth"], str) else rec["ground_truth"]
    gt_parse = gt_raw.get("gt_parse", gt_raw)
    gt = convert_gt(gt_parse)

    # Save image
    img_path = os.path.join(IMG_DIR, f"{doc_id}.png")
    img = rec["image"]
    if isinstance(img, PILImage.Image):
        img.save(img_path)
    elif isinstance(img, bytes):
        with open(img_path, "wb") as f:
            f.write(img)

    samples.append({
        "doc_id": doc_id,
        "image_path": img_path,
        "source_index": idx,
        "ground_truth": gt,
    })

# Count non-N/A fields
filled = Counter()
for s in samples:
    for c in ALL_CONCEPTS:
        if s["ground_truth"].get(c, "N/A") != "N/A":
            filled[c] += 1

print(f"Sampled {len(samples)} W-2 forms")
print(f"\nField fill rates (top 20):")
for concept, count in filled.most_common(20):
    print(f"  {concept:<30s}: {count}/{SAMPLE_SIZE} ({count/SAMPLE_SIZE:.0%})")
print(f"\nAlways empty (never filled): {sum(1 for c in ALL_CONCEPTS if filled[c] == 0)}")

Sampled 50 W-2 forms

Field fill rates (top 20):
  employee_ssn                  : 50/50 (100%)
  employer_ein                  : 50/50 (100%)
  employer_name                 : 50/50 (100%)
  employee_name                 : 50/50 (100%)
  control_number                : 50/50 (100%)
  employer_address              : 50/50 (100%)
  employer_city_state_zip       : 50/50 (100%)
  employee_address              : 50/50 (100%)
  employee_city_state_zip       : 50/50 (100%)
  wages_tips_compensation       : 50/50 (100%)
  social_security_wages         : 50/50 (100%)
  medicare_wages                : 50/50 (100%)
  social_security_tips          : 50/50 (100%)
  allocated_tips                : 50/50 (100%)
  federal_tax_withheld          : 50/50 (100%)
  social_security_tax           : 50/50 (100%)
  medicare_tax                  : 50/50 (100%)
  dependent_care_benefits       : 50/50 (100%)
  nonqualified_plans            : 50/50 (100%)
  box_12a_code                  : 50/50 (100%)

Always emp

In [5]:
def normalize_value(v):
    """Normalize a value for comparison."""
    v = str(v).strip().lower()
    v = v.replace("$", "").replace(",", "").strip()
    v = " ".join(v.split())
    try:
        numeric = float(v)
        v = f"{numeric:.2f}"
    except ValueError:
        pass
    return v

def score_document(gt, pred):
    """Score all concepts with GT values using value-first CBA matching."""
    pred_norm = {k: normalize_value(v) for k, v in pred.items()}
    pred_values = {v for v in pred_norm.values() if v != normalize_value("N/A")}

    per_field = {}
    field_correct = 0
    cba_correct = 0
    scored = []

    for concept in ALL_CONCEPTS:
        gt_val = normalize_value(gt.get(concept, "N/A"))
        pred_val = normalize_value(pred.get(concept, "N/A"))

        # Skip concepts without GT
        if gt_val == normalize_value("N/A"):
            continue

        scored.append(concept)

        cba_match = (gt_val == pred_val)
        field_match = (gt_val in pred_values)

        if cba_match:
            cba_correct += 1
        if field_match:
            field_correct += 1

        bound_to = None
        if field_match and not cba_match:
            for k, v in pred.items():
                if normalize_value(v) == gt_val and k != concept:
                    bound_to = k
                    break

        per_field[concept] = {
            "gt_value": gt_val,
            "pred_value": pred_val,
            "field_match": field_match,
            "cba_match": cba_match,
            "misbinding": field_match and not cba_match,
            "bound_to": bound_to,
        }

    total = len(scored)
    if total == 0:
        return {"field_recall": 0, "cba_strict": 0, "delta": 0, "total": 0, "per_field": {}, "misbinding_count": 0}

    misbinding_count = sum(1 for d in per_field.values() if d["misbinding"])

    # CBA-soft: same family = 0.5 credit for misbindings
    cba_soft_score = 0
    for concept, detail in per_field.items():
        if detail["cba_match"]:
            cba_soft_score += 1.0
        elif detail["misbinding"] and detail["bound_to"]:
            gt_fam = CONCEPT_TO_FAMILY.get(concept, "")
            pred_fam = CONCEPT_TO_FAMILY.get(detail["bound_to"], "")
            cba_soft_score += 0.5 if gt_fam == pred_fam else 0.0

    return {
        "field_recall": field_correct / total,
        "cba_strict": cba_correct / total,
        "cba_soft": cba_soft_score / total,
        "delta": (field_correct - cba_correct) / total,
        "total": total,
        "scored_concepts": scored,
        "per_field": per_field,
        "misbinding_count": misbinding_count,
    }

print(f"Scoring functions ready ({len(ALL_CONCEPTS)} concepts)")

Scoring functions ready (44 concepts)


In [6]:
def encode_image(image_path, max_width=1024):
    """Load, resize, and base64-encode an image."""
    img = PILImage.open(image_path)
    if img.width > max_width:
        ratio = max_width / img.width
        img = img.resize((max_width, int(img.height * ratio)), PILImage.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")

def extract_anthropic(image_path, model_id):
    """Extract via Anthropic vision API."""
    b64 = encode_image(image_path)
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    message = client.messages.create(
        model=model_id,
        max_tokens=2048,
        system=W2_SYSTEM_PROMPT,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}},
                {"type": "text", "text": "Extract all fields from this W-2 tax form image."},
            ],
        }],
    )
    raw = message.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)

def extract_openai(image_path, model_id):
    """Extract via OpenAI vision API."""
    b64 = encode_image(image_path)
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model=model_id,
        max_tokens=2048,
        messages=[
            {"role": "system", "content": W2_SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": "Extract all fields from this W-2 tax form image."},
            ]},
        ],
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)

MODELS = {
    "haiku": {"provider": "anthropic", "model_id": "claude-haiku-4-5-20251001"},
    "gpt4o-mini": {"provider": "openai", "model_id": "gpt-4o-mini"},
}

def extract_w2(image_path, model_name):
    """Route to correct provider."""
    cfg = MODELS[model_name]
    if cfg["provider"] == "anthropic":
        return extract_anthropic(image_path, cfg["model_id"])
    else:
        return extract_openai(image_path, cfg["model_id"])

print(f"Extraction functions ready. Models: {list(MODELS.keys())}")

Extraction functions ready. Models: ['haiku', 'gpt4o-mini']


In [7]:
all_results = {}

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Running: {model_name} ({MODELS[model_name]['model_id']})")
    print(f"{'='*60}")

    results = []
    errors = []

    for i, s in enumerate(samples):
        predicted = None
        for attempt in range(5):
            try:
                predicted = extract_w2(s["image_path"], model_name)
                break
            except Exception as e:
                err_str = str(e)
                is_rate_limit = "429" in err_str or "rate_limit" in err_str.lower()
                if attempt < 4:
                    if is_rate_limit:
                        wait = min(10 * (2 ** attempt), 120)  # 10, 20, 40, 80s for rate limits
                        print(f"  [{s['doc_id']}] Rate limited (attempt {attempt+1}). Waiting {wait}s...")
                    else:
                        wait = 2 ** (attempt + 1)
                        print(f"  [{s['doc_id']}] Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"  [{s['doc_id']}] FAILED after 5 attempts: {e}")
                    errors.append({"doc_id": s["doc_id"], "error": str(e)})

        if predicted is None:
            continue

        scores = score_document(s["ground_truth"], predicted)
        results.append({
            "doc_id": s["doc_id"],
            "predicted": predicted,
            "ground_truth": s["ground_truth"],
            "scores": scores,
        })

        mb = scores["misbinding_count"]
        status = "OK" if mb == 0 else f"MISBIND={mb}"
        if (i + 1) % 10 == 0 or i == 0 or mb > 0:
            print(f"  [{i+1:2d}/{len(samples)}] {s['doc_id']} — F1={scores['field_recall']:.3f}  CBA={scores['cba_strict']:.3f}  scored={scores['total']}  {status}")

        # Longer sleep for OpenAI to avoid TPM rate limits on vision requests
        time.sleep(1.5 if MODELS[model_name]["provider"] == "anthropic" else 5.0)

    # Aggregate
    if results:
        avg_f1 = sum(r["scores"]["field_recall"] for r in results) / len(results)
        avg_cba = sum(r["scores"]["cba_strict"] for r in results) / len(results)
        avg_cba_soft = sum(r["scores"]["cba_soft"] for r in results) / len(results)
        avg_delta = sum(r["scores"]["delta"] for r in results) / len(results)
        total_misbindings = sum(r["scores"]["misbinding_count"] for r in results)
    else:
        avg_f1 = avg_cba = avg_cba_soft = avg_delta = total_misbindings = 0

    all_results[model_name] = {
        "results": results,
        "errors": errors,
        "avg_f1": round(avg_f1, 4),
        "avg_cba": round(avg_cba, 4),
        "avg_cba_soft": round(avg_cba_soft, 4),
        "avg_delta": round(avg_delta, 4),
        "total_misbindings": total_misbindings,
    }

    print(f"\n--- {model_name} Summary ---")
    print(f"  Field Recall:     {avg_f1:.4f}")
    print(f"  CBA-strict:   {avg_cba:.4f}")
    print(f"  CBA-soft:     {avg_cba_soft:.4f}")
    print(f"  Delta:        {avg_delta:.4f}")
    print(f"  Misbindings:  {total_misbindings}")
    print(f"  Errors:       {len(errors)}")

print(f"\n{'='*60}")
print("ALL EXPERIMENTS COMPLETE")
print(f"{'='*60}")


Running: haiku (claude-haiku-4-5-20251001)
  [ 1/50] w2_0000 — F1=0.447  CBA=0.395  scored=38  MISBIND=2
  [ 2/50] w2_0001 — F1=0.526  CBA=0.500  scored=38  MISBIND=1
  [ 3/50] w2_0002 — F1=0.316  CBA=0.132  scored=38  MISBIND=7
  [ 4/50] w2_0003 — F1=0.421  CBA=0.237  scored=38  MISBIND=7
  [ 5/50] w2_0004 — F1=0.395  CBA=0.237  scored=38  MISBIND=6
  [ 6/50] w2_0005 — F1=0.342  CBA=0.158  scored=38  MISBIND=7
  [ 7/50] w2_0006 — F1=0.421  CBA=0.289  scored=38  MISBIND=5
  [ 8/50] w2_0007 — F1=0.368  CBA=0.263  scored=38  MISBIND=4
  [ 9/50] w2_0008 — F1=0.289  CBA=0.263  scored=38  MISBIND=1
  [10/50] w2_0009 — F1=0.289  CBA=0.026  scored=38  MISBIND=10
  [11/50] w2_0010 — F1=0.605  CBA=0.447  scored=38  MISBIND=6
  [12/50] w2_0011 — F1=0.553  CBA=0.395  scored=38  MISBIND=6
  [13/50] w2_0012 — F1=0.500  CBA=0.395  scored=38  MISBIND=4
  [14/50] w2_0013 — F1=0.289  CBA=0.184  scored=38  MISBIND=4
  [15/50] w2_0014 — F1=0.368  CBA=0.211  scored=38  MISBIND=6
  [16/50] w2_0015 — F1=0.

In [12]:
# Cross-model comparison
print("=" * 75)
print("CROSS-MODEL COMPARISON — W-2 Tax Forms")
print("=" * 75)

print(f"\n{'Model':<15} {'Field Recall':>10} {'CBA-strict':>12} {'CBA-soft':>10} {'Delta':>8} {'Misbindings':>13} {'Errors':>8}")
print("-" * 80)
for name, res in all_results.items():
    print(f"{name:<15} {res['avg_f1']:>10.4f} {res['avg_cba']:>12.4f} {res['avg_cba_soft']:>10.4f} {res['avg_delta']:>8.4f} {res['total_misbindings']:>13} {len(res['errors']):>8}")

total_misbindings = sum(r["total_misbindings"] for r in all_results.values())
print(f"\nTotal misbindings across all models: {total_misbindings}")

# Per-concept accuracy
print(f"\n{'='*75}")
print("PER-CONCEPT ACCURACY (across all models, GT-backed concepts only)")
print(f"{'='*75}")

concept_stats = defaultdict(lambda: {"cba": 0, "field": 0, "total": 0, "misbindings": 0})
for model_name, res in all_results.items():
    for r in res["results"]:
        for concept, detail in r["scores"]["per_field"].items():
            cs = concept_stats[concept]
            cs["total"] += 1
            if detail["cba_match"]:
                cs["cba"] += 1
            if detail["field_match"]:
                cs["field"] += 1
            if detail["misbinding"]:
                cs["misbindings"] += 1

# Sort by misbinding count (most problematic first)
concept_rows = []
for concept in ALL_CONCEPTS:
    cs = concept_stats[concept]
    if cs["total"] == 0:
        continue
    concept_rows.append((concept, cs))

concept_rows.sort(key=lambda x: -x[1]["misbindings"])

print(f"\n{'Concept':<30} {'Family':<15} {'Field Acc':>10} {'CBA Acc':>10} {'Delta':>8} {'Misbindings':>13}")
print("-" * 90)
for concept, cs in concept_rows[:25]:  # Top 25
    f_acc = cs["field"] / cs["total"]
    c_acc = cs["cba"] / cs["total"]
    delta = f_acc - c_acc
    fam = CONCEPT_TO_FAMILY[concept]
    flag = " ***" if cs["misbindings"] > 5 else ""
    print(f"{concept:<30} {fam:<15} {f_acc:>10.3f} {c_acc:>10.3f} {delta:>8.3f} {cs['misbindings']:>13}{flag}")

CROSS-MODEL COMPARISON — W-2 Tax Forms

Model             Field F1   CBA-strict   CBA-soft    Delta   Misbindings   Errors
--------------------------------------------------------------------------------
haiku               0.4200       0.2921     0.3479   0.1279           243        0
gpt4o-mini          0.5868       0.4932     0.5297   0.0937           178        0

Total misbindings across all models: 421

PER-CONCEPT ACCURACY (across all models, GT-backed concepts only)

Concept                        Family           Field Acc    CBA Acc    Delta   Misbindings
------------------------------------------------------------------------------------------
social_security_tips           compensation         0.790      0.010    0.780            78 ***
allocated_tips                 compensation         0.450      0.010    0.440            44 ***
local_1_income_tax             state_local          0.650      0.360    0.290            29 ***
local_2_income_tax             state_local       

In [11]:
# Misbinding confusion pairs
print("=" * 75)
print("MISBINDING CONFUSION PAIRS")
print("=" * 75)

confusion = Counter()
all_misbindings = []
for model_name, res in all_results.items():
    for r in res["results"]:
        for concept, detail in r["scores"]["per_field"].items():
            if detail["misbinding"] and detail["bound_to"]:
                confusion[(concept, detail["bound_to"])] += 1
                all_misbindings.append({
                    "model": model_name,
                    "doc_id": r["doc_id"],
                    "concept": concept,
                    "bound_to": detail["bound_to"],
                    "gt_value": detail["gt_value"],
                    "family_gt": CONCEPT_TO_FAMILY.get(concept, "?"),
                    "family_pred": CONCEPT_TO_FAMILY.get(detail["bound_to"], "?"),
                })

if confusion:
    print(f"\n{'Expected Concept':<30} {'Bound To':<30} {'Count':>6} {'Families':>25}")
    print("-" * 95)
    for (src, dst), count in confusion.most_common(20):
        fam_src = CONCEPT_TO_FAMILY.get(src, "?")
        fam_dst = CONCEPT_TO_FAMILY.get(dst, "?")
        same = "SAME-FAM" if fam_src == fam_dst else "CROSS-FAM"
        print(f"{src:<30} {dst:<30} {count:>6} {fam_src}->{fam_dst} ({same})")

    # Family-level summary
    print(f"\n{'='*75}")
    print("FAMILY-LEVEL CONFUSION SUMMARY")
    print(f"{'='*75}")
    fam_confusion = Counter()
    for mb in all_misbindings:
        fam_confusion[(mb["family_gt"], mb["family_pred"])] += 1

    print(f"\n{'GT Family':<20} {'Pred Family':<20} {'Count':>6} {'Type':>12}")
    print("-" * 60)
    for (fgt, fpred), count in fam_confusion.most_common():
        same = "within-fam" if fgt == fpred else "cross-fam"
        print(f"{fgt:<20} {fpred:<20} {count:>6} {same:>12}")
else:
    print("No misbindings detected — W-2 fixed layout works as expected!")

# W-2 as control condition assessment
print(f"\n{'='*75}")
print("W-2 AS CONTROL CONDITION")
print(f"{'='*75}")
print(f"\nTotal misbindings: {len(all_misbindings)}")
for model_name, res in all_results.items():
    delta = res["avg_delta"]
    mb = res["total_misbindings"]
    print(f"  {model_name}: delta={delta:.4f}, misbindings={mb}")
avg_delta_all = sum(r["avg_delta"] for r in all_results.values()) / len(all_results)
if avg_delta_all < 0.05:
    print(f"\nAvg delta ({avg_delta_all:.4f}) < 0.05 threshold")
    print("CONFIRMED: W-2 fixed layout shows minimal misbinding, validating CBA metric")
else:
    print(f"\nAvg delta ({avg_delta_all:.4f}) >= 0.05")
    print("UNEXPECTED: Even W-2 fixed layout shows significant misbinding — investigate confusions above")

MISBINDING CONFUSION PAIRS

Expected Concept               Bound To                        Count                  Families
-----------------------------------------------------------------------------------------------
social_security_tips           social_security_wages              69 compensation->compensation (SAME-FAM)
allocated_tips                 medicare_wages                     24 compensation->compensation (SAME-FAM)
employee_ssn                   employer_ein                       21 identity->identity (SAME-FAM)
state_2                        state_1                            20 state_local->state_local (SAME-FAM)
state_2_wages                  state_1_wages                      17 state_local->state_local (SAME-FAM)
local_1_income_tax             state_1_income_tax                 13 state_local->state_local (SAME-FAM)
state_2_income_tax             state_1_income_tax                 11 state_local->state_local (SAME-FAM)
employer_ein                   employee_ssn     

In [10]:
export = {
    "experiment": "hindsight_w2_cba",
    "domain": "tax_forms",
    "dataset": "singhsays/fake-w2-us-tax-form-dataset",
    "sample_size": len(samples),
    "total_concepts": len(ALL_CONCEPTS),
    "ontology": W2_ONTOLOGY,
    "models": {k: v for k, v in MODELS.items()},
    "scoring_method": "value-first CBA, GT-backed concepts only",
    "results": {
        model_name: {
            "field_recall": res["avg_f1"],
            "cba_strict": res["avg_cba"],
            "cba_soft": res["avg_cba_soft"],
            "delta": res["avg_delta"],
            "total_misbindings": res["total_misbindings"],
            "num_errors": len(res["errors"]),
        }
        for model_name, res in all_results.items()
    },
    "confusion_pairs": [
        {"expected": src, "bound_to": dst, "count": count}
        for (src, dst), count in confusion.most_common(20)
    ] if confusion else [],
    "total_misbindings": total_misbindings,
}

output_path = "/tmp/hindsight_w2_cba_results.json"
with open(output_path, "w") as f:
    json.dump(export, f, indent=2)
print(f"Results exported to: {output_path}")

# Cross-domain comparison
print(f"\n{'='*75}")
print("CROSS-DOMAIN COMPARISON (all experiments)")
print(f"{'='*75}")
print(f"\n{'Domain':<20} {'Dataset':<20} {'Misbindings':>12} {'Haiku Delta':>13} {'GPT4o-mini Delta':>17}")
print("-" * 85)
print(f"{'Paystubs':<20} {'Synthetic':<20} {'~5':>12} {'~0':>13} {'~0':>17}")
print(f"{'Receipts (CORD)':<20} {'CORD v2':<20} {'48':>12} {'-0.10':>13} {'~0':>17}")

haiku_d = all_results.get("haiku", {}).get("avg_delta", 0)
gpt_d = all_results.get("gpt4o-mini", {}).get("avg_delta", 0)
total_mb = sum(r["total_misbindings"] for r in all_results.values())
print(f"{'W-2 Tax Forms':<20} {'Synthetic W-2':<20} {total_mb:>12} {haiku_d:>13.3f} {gpt_d:>17.3f}")

print(f"{'Employment Law':<20} {'LEDGAR':<20} {'350':>12} {'+0.207':>13} {'+0.260':>17}")
print(f"{'Commercial Law':<20} {'CUAD v1':<20} {'434':>12} {'+0.244':>13} {'+0.335':>17}")

Results exported to: /tmp/hindsight_w2_cba_results.json

CROSS-DOMAIN COMPARISON (all experiments)

Domain               Dataset               Misbindings   Haiku Delta  GPT4o-mini Delta
-------------------------------------------------------------------------------------
Paystubs             Synthetic                      ~5            ~0                ~0
Receipts (CORD)      CORD v2                        48         -0.10                ~0
W-2 Tax Forms        Synthetic W-2                 421         0.128             0.094
Employment Law       LEDGAR                        350        +0.207            +0.260
Commercial Law       CUAD v1                       434        +0.244            +0.335
